# PyTensor rewrite exploration

Goal: see what ops end up in the post-rewrite graphs of PTGP's objectives. We want to confirm that `pt.linalg.solve` and `slogdet` get lowered to `Cholesky` + `SolveTriangular`, and that no `MatrixInverse` / generic `Solve` / `SLogDet` survives.

This notebook is a scratch pad for understanding baseline behavior before codifying assertions in `tests/test_rewrites.py`.

In [1]:
from collections import Counter

import numpy as np
import pymc as pm
import pytensor
import pytensor.tensor as pt
from pytensor.compile.mode import Mode
from pytensor.graph.basic import ancestors
from pytensor.graph.rewriting.utils import rewrite_graph

import ptgp as pg

# Force the C linker so rewrites match what we'd use in production;
# the default "auto" linker picks numba on this machine and some
# inplace Composite ops aren't numba-dispatched yet.
FAST_RUN_C = Mode(linker="cvm", optimizer="fast_run")

/var/folders/nw/c4qfxc81785fbrlh7pkwk9mh0000gn/T/ipykernel_84725/699138501.py:8: FutureWarning: `pytensor.graph.basic.ancestors` was moved to `pytensor.graph.traversal.ancestors`. Calling it from the old location will fail in a future release.
  from pytensor.graph.basic import ancestors


## Helpers

`op_counts` walks either a symbolic `Variable` (pre- or post-rewrite) or a compiled `pytensor.function` and returns a `Counter` of op class names.

In [2]:
def op_counts(obj):
    """Count op types in a pytensor graph (Variable, list of Variables, or compiled Function)."""
    if hasattr(obj, "maker"):  # compiled Function
        nodes = obj.maker.fgraph.apply_nodes
    else:
        roots = obj if isinstance(obj, (list, tuple)) else [obj]
        nodes = {v.owner for v in ancestors(roots) if v.owner is not None}
    return Counter(type(n.op).__name__ for n in nodes)


def summary(counts, interesting=(
    "Cholesky", "SolveTriangular", "Solve", "SolveBase",
    "MatrixInverse", "Inv", "SLogDet", "Det", "Dot", "Dot22", "BlockDiag",
)):
    """Pretty-print only the ops we care about for rewrite verification."""
    for name in interesting:
        if name in counts:
            print(f"  {name}: {counts[name]}")

## Toy data

Small enough that graph dumps stay readable; we only care about structure.

In [3]:
rng = np.random.default_rng(0)
N, M = 20, 6
X_np = np.sort(rng.uniform(0, 5, N))[:, None]
y_np = np.sin(X_np[:, 0]) + 0.1 * rng.standard_normal(N)
Z_np = np.linspace(0, 5, M)[:, None]

X_var = pt.matrix("X")
y_var = pt.vector("y")
Z_var = pt.matrix("Z")

## GP: marginal log likelihood

Expected: one solve and one log-det on `K_ff + σ²I`. Post-rewrite we want `Cholesky` + `SolveTriangular`.

In [4]:
with pm.Model() as gp_model:
    ls = pm.HalfFlat("ls")
    eta = pm.Exponential("eta", lam=1.0)
    sigma = pm.HalfNormal("sigma", sigma=1.0)
    gp = pg.gp.Unapproximated(
        kernel=eta**2 * pg.kernels.Matern52(input_dim=1, ls=ls),
        likelihood=pg.likelihoods.Gaussian(sigma=sigma),
    )

mll = pg.objectives.marginal_log_likelihood(gp, X_var, y_var)
print("pre-rewrite:")
summary(op_counts(mll))

pre-rewrite:


In [5]:
mll_opt = rewrite_graph(mll, include=("fast_run",))
print("post-rewrite (rewrite_graph, fast_run):")
summary(op_counts(mll_opt))

/Users/bill/miniconda3/envs/ptgp/lib/python3.14/site-packages/pytensor/graph/rewriting/basic.py:110: UserWarning: A Supervisor feature is missing from FunctionGraph(Composite{(-0.5 * (i0 + i1 + (1.8378770664093453 * i2)))}(Squeeze{axis=0}(CGemv{no_inplace}(AllocEmpty{dtype='float64'}(1), 1.0, MatrixTranspose(*6 -> Sub(ExpandDims{axis=1}(y), [[0.]])), Squeeze{axis=1}(Blockwise{Solve{assume_a='gen', lower=False, b_ndim=2, overwrite_a=False, overwrite_b=False}, (m,m),(m,n)->(m,n)}(MatrixTranspose(*4 -> Add(SpecifyAssumptions{assumptions=frozenset({'symmetric', 'positive_definite'})}(Mul(SpecifyAssumptions{assumptions=frozenset({'symmetric', 'positive_definite'})}(Composite{...}(ExpandDims{axis=0}(*7 -> Sqr(*1 -> Squeeze{axis=1}(True_div(MatrixTranspose(AdvancedSubtensor1(MatrixTranspose(X), [0])), ExpandDims{axes=(0, 1)}(half_flat_rv{"->()"}(RNG(<Generator(PCG64) at 0x130A4AF80>), NoneConst{None})))))), ExpandDims{axis=1}(*7), CGer{non-destructive}(Alloc(0.0, *2 -> Composite{switch(lt(0, 

post-rewrite (rewrite_graph, fast_run):
  Solve: 1
  SLogDet: 1


In [6]:
value_vars = list(gp_model.value_vars)
fn_mll = pytensor.function(
    [X_var, y_var, *value_vars], mll,
    mode=FAST_RUN_C, on_unused_input="ignore", accept_inplace=True,
)
print("post-rewrite (compiled, FAST_RUN):")
summary(op_counts(fn_mll))

post-rewrite (compiled, FAST_RUN):
  Solve: 1
  SLogDet: 1


In [7]:
# Full dprint of the compiled graph — scan for any lingering MatrixInverse / generic Solve / SLogDet.
pytensor.dprint(fn_mll)

Composite{(-0.5 * (i0 + i1 + (1.8378770664093453 * i2)))} [id A] 40
 ├─ Squeeze{axis=0} [id B] 39
 │  └─ CGemv{inplace} [id C] 38
 │     ├─ AllocEmpty{dtype='float64'} [id D] 5
 │     │  └─ 1 [id E]
 │     ├─ 1.0 [id F]
 │     ├─ MatrixTranspose [id G] 16
 │     │  └─ Sub [id H] 7
 │     │     ├─ ExpandDims{axis=1} [id I] 0
 │     │     │  └─ y [id J]
 │     │     └─ [[0.]] [id K]
 │     ├─ Squeeze{axis=1} [id L] 37
 │     │  └─ Solve{assume_a='gen', lower=False, b_ndim=2, overwrite_a=False, overwrite_b=False} [id M] 36
 │     │     ├─ Add [id N] 34
 │     │     │  ├─ SpecifyAssumptions{assumptions=frozenset({'symmetric', 'positive_definite'})} [id O] 32
 │     │     │  │  └─ Mul [id P] 31
 │     │     │  │     ├─ SpecifyAssumptions{assumptions=frozenset({'symmetric', 'positive_definite'})} [id Q] 30
 │     │     │  │     │  └─ Composite{...} [id R] 29
 │     │     │  │     │     ├─ ExpandDims{axis=0} [id S] 26
 │     │     │  │     │     │  └─ Composite{...}.0 [id T] 22
 │     │     │

## VFE: collapsed ELBO

In [8]:
with pm.Model() as vfe_model:
    ls = pm.HalfFlat("ls")
    eta = pm.Exponential("eta", lam=1.0)
    sigma = pm.HalfNormal("sigma", sigma=1.0)
    vfe = pg.gp.VFE(
        kernel=eta**2 * pg.kernels.Matern52(input_dim=1, ls=ls),
        likelihood=pg.likelihoods.Gaussian(sigma=sigma),
        inducing_variable=pg.inducing.Points(Z_var),
    )

elbo_vfe = pg.objectives.collapsed_elbo(vfe, X_var, y_var)
print("pre-rewrite:")
summary(op_counts(elbo_vfe))

elbo_vfe_opt = rewrite_graph(elbo_vfe, include=("fast_run",))
print("\npost-rewrite (rewrite_graph):")
summary(op_counts(elbo_vfe_opt))

value_vars_vfe = list(vfe_model.value_vars)
fn_vfe = pytensor.function(
    [X_var, y_var, Z_var, *value_vars_vfe], elbo_vfe,
    mode=FAST_RUN_C, on_unused_input="ignore", accept_inplace=True,
)
print("\npost-rewrite (compiled):")
summary(op_counts(fn_vfe))

pre-rewrite:


/Users/bill/miniconda3/envs/ptgp/lib/python3.14/site-packages/pytensor/graph/rewriting/basic.py:110: UserWarning: A Supervisor feature is missing from FunctionGraph(Composite{((-0.5 * (i0 + i1 + (1.8378770664093453 * i2))) + ((-0.5 * i3) / sqr(i4)))}(Squeeze{axis=0}(CGemv{no_inplace}(AllocEmpty{dtype='float64'}(1), 1.0, MatrixTranspose(*2 -> Sub(ExpandDims{axis=1}(y), [[0.]])), Squeeze{axis=1}(Blockwise{Solve{assume_a='gen', lower=False, b_ndim=2, overwrite_a=False, overwrite_b=False}, (m,m),(m,n)->(m,n)}(MatrixTranspose(*15 -> Gemm{no_inplace}(AdvancedSetSubtensor(Alloc(0.0, *13 -> Shape_i{0}(X), *13), Mul(Sqr(ExpandDims{axis=0}(*4 -> halfnormal_rv{"(),()->()"}(RNG(<Generator(PCG64) at 0x1319D8F20>), NoneConst{None}, 0.0, 1.0))), ExtractDiag{offset=0, axis1=0, axis2=1, view=True}(Eye{dtype='float64'}(*13, *13, 0))), *14 -> ARange{dtype='int64'}(0, *13, 1), *14), 1.0, MatrixTranspose(*10 -> Composite{...}(*1 -> Sqr(ExpandDims{axes=(0, 1)}(exponential_rv{"()->()"}(RNG(<Generator(PCG64) 


post-rewrite (rewrite_graph):
  Cholesky: 1
  Solve: 1
  SLogDet: 1

post-rewrite (compiled):
  Cholesky: 1
  Solve: 1
  SLogDet: 1


## SVGP: ELBO

In [9]:
q_mu_var = pt.vector("q_mu")
q_sqrt_var = pt.matrix("q_sqrt")

with pm.Model() as svgp_model:
    ls = pm.HalfFlat("ls")
    eta = pm.Exponential("eta", lam=1.0)
    svgp = pg.gp.SVGP(
        kernel=eta**2 * pg.kernels.Matern52(input_dim=1, ls=ls),
        likelihood=pg.likelihoods.Gaussian(sigma=0.1),
        inducing_variable=pg.inducing.Points(Z_var),
        q_mu=q_mu_var,
        q_sqrt=q_sqrt_var,
    )

elbo_svgp = pg.objectives.elbo(svgp, X_var, y_var)
print("pre-rewrite:")
summary(op_counts(elbo_svgp))

elbo_svgp_opt = rewrite_graph(elbo_svgp, include=("fast_run",))
print("\npost-rewrite (rewrite_graph):")
summary(op_counts(elbo_svgp_opt))

value_vars_svgp = list(svgp_model.value_vars)
fn_svgp = pytensor.function(
    [X_var, y_var, Z_var, q_mu_var, q_sqrt_var, *value_vars_svgp],
    elbo_svgp, mode=FAST_RUN_C, on_unused_input="ignore", accept_inplace=True,
)
print("\npost-rewrite (compiled):")
summary(op_counts(fn_svgp))

pre-rewrite:


/Users/bill/miniconda3/envs/ptgp/lib/python3.14/site-packages/pytensor/graph/rewriting/basic.py:110: UserWarning: A Supervisor feature is missing from FunctionGraph(Composite{((-0.5 * i0) - (0.5 * (((i3 + i4) - i2) - i1)))}(Sum{axes=None}(Composite{(-2.7672931195787456 + (99.99999999999999 * (sqr(i1) + (i2 - i3) + i0)))}(CAReduce{Composite{(i0 + sqr(i1))}, axis=1}(Dot22(*12 -> MatrixTranspose(*7 -> Blockwise{SolveTriangular{unit_diagonal=False, lower=True, b_ndim=2, overwrite_b=False}, (m,m),(m,n)->(m,n)}(Blockwise{Cholesky{lower=True, overwrite_a=False}, (m,m)->(m,m)}(SpecifyAssumptions{assumptions=frozenset({'symmetric', 'positive_definite'})}(Mul(SpecifyAssumptions{assumptions=frozenset({'symmetric', 'positive_definite'})}(Composite{...}(ExpandDims{axis=0}(*9 -> Sqr(*2 -> Squeeze{axis=1}(True_div(MatrixTranspose(AdvancedSubtensor1(MatrixTranspose(Z), [0])), *4 -> ExpandDims{axes=(0, 1)}(half_flat_rv{"->()"}(RNG(<Generator(PCG64) at 0x136A0CC80>), NoneConst{None})))))), *14 -> Expand


post-rewrite (rewrite_graph):
  Cholesky: 1
  SolveTriangular: 1
  SLogDet: 1
  Dot22: 2

post-rewrite (compiled):
  Cholesky: 1
  SolveTriangular: 1
  SLogDet: 1
  Dot22: 2


## Scratch: iterate on rewrites interactively

`rewrite_graph` accepts `include=` / `exclude=` / `custom_rewrite=` — use this to probe *which* rewrite is responsible for a particular lowering (e.g. disable `linalg` rewrites to confirm that's what replaces `Solve` with `SolveTriangular`).

In [10]:
# Example: strip linalg rewrites and see what survives.
# mll_no_linalg = rewrite_graph(mll, include=("fast_run",), exclude=("linalg",))
# summary(op_counts(mll_no_linalg))